# STC Jawwy

In [ ]:
"""
Here we install libraries that are not installed by default
Example:  pyslsb
Feel free to add any library you are planning to use.
"""
!pip install pyxlsb

In [ ]:
# Import the required libraries
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd     # provides high-performance, easy to use structures and data analysis tools
import pyxlsb           # Excel extention to read xlsb files (the input file)
import numpy as np      # provides fast mathematical computation on arrays and matrices
import plotly.express as px

# Jawwy dataset
The dataset consists of meta details about the movies and tv shows as genre.
Also details about Users activities, spent duration and if watching in High definition or standard definition.
You have to analyse this dataset to find top insights, findings and to solve the four tasks assigned to you.

In [ ]:
dataframe = pd.read_excel('data/stc_tv_clean.xlsx',sheet_name="Final_Dataset")
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
# you can use   df=dataframe.copy()

In [ ]:
# check the data shape
dataframe.shape

In [ ]:
# display the first 5 rows
dataframe.head()

In [ ]:
# Data Preprocessing on the input data
#dataframe = dataframe.drop(columns=['Column1'])         # dropping the index column
dataframe['program_name'] = dataframe['program_name'].str.strip()  # trim spaces in movies names to avoid misspellings in input data
# The date column seems to be parsed correctly by pd.read_excel, removing the explicit conversion that caused overflow.
# origin = pd.Timestamp('1899-12-30')  # Excel origin
# dataframe['date_'] = origin + pd.to_timedelta(pd.to_numeric(dataframe['date_'], errors='coerce'), unit='D')
dataframe[['duration_seconds', 'season','episode','series_title','hd']] = dataframe[['duration_seconds', 'season','episode','series_title','hd']].apply(pd.to_numeric)  # read numeric columns as numeric data types
dataframe[['user_id_maped', 'program_name','program_class','program_desc','program_genre','original_name']] = dataframe[['user_id_maped', 'program_name','program_class','program_desc','program_genre','original_name']].astype(str) # read string columns as string data types

In [ ]:
# display the dataset after applying data types
dataframe.head()

In [ ]:
dataframe.columns

In [ ]:
# describe the numeric values in the dataset
dataframe.describe()

In [ ]:
# check if any column has null value in the dataset
dataframe.isnull().any()

# Task 1
##### You are required to work on task one to study and HD flag for available dataset

In [ ]:
dataframe.value_counts()

In [ ]:
# make a copy of the dataframe for working on task 1
df=dataframe.copy()

In [ ]:
df['hd'].value_counts(dropna=False)



In [ ]:
hd_pct = (df['hd'].value_counts(normalize=True) * 100).round(2)
hd_pct
#النسبة المشاهداة المئوية HD vs SD (1 = HD  0 = SD)

In [ ]:
hd_pct.rename(index={0:'SD',1:'HD'})

In [ ]:
(df['hd']
 .replace({0:'SD', 1:'HD'})
 .value_counts(normalize=True)
 .mul(100)
 .round(2)
 .plot(kind='bar', title='HD vs SD (%)', ylabel='Percentage'))


In [ ]:
mean_minutes = (df.groupby('hd')['duration_seconds'].mean()/60).round(2)
mean_minutes.rename(index={0:'SD', 1:'HD'})
#توسط  مدة المشاهدة لكل جودة (بالدقائق)

In [ ]:
median_minutes = (df.groupby('hd')['duration_seconds'].median()/60).round(2)
median_minutes.rename(index={0:'SD', 1:'HD'})
# وسيط

In [ ]:
total_hours = (df.groupby('hd')['duration_seconds'].sum()/3600).round(2)
total_hours.rename(index={0:'SD', 1:'HD'})


In [ ]:
# جدول ملخص للاوامر السابقه
summary = df.groupby('hd').agg(
    views=('hd','size'),
    users=('user_id_maped','nunique'),
    mean_min=('duration_seconds', lambda s: round(s.mean()/60,2)),
    median_min=('duration_seconds', lambda s: round(s.median()/60,2)),
    total_hours=('duration_seconds', lambda s: round(s.sum()/3600,2))
)

summary.index = summary.index.map({0:'SD', 1:'HD'})
summary


استنتاج:

المحتوى SD هو الأكثر استهلاكاً فعلياً

مستخدمو HD أكثر عدداً لكن تفاعلهم ضعيف

HD لديه معدل انسحاب سريع جداً

المستخدم يميل إلى الاستمرار  في المشاهده SD أكثر

In [ ]:
df['quality'] = df['hd'].map({0:'SD', 1:'HD'})
df['minutes'] = df['duration_seconds'] / 60
df['hours'] = df['duration_seconds'] / 3600


In [ ]:

df['quality'] = df['hd'].map({0:'SD', 1:'HD'})
df['minutes'] = df['duration_seconds'] / 60
df['hours']   = df['duration_seconds'] / 3600

quality_summary = (df.groupby('quality')
                     .agg(views=('quality','size'),
                          users=('user_id_maped','nunique'),
                          mean_min=('minutes','mean'),
                          median_min=('minutes','median'),
                          total_hours=('hours','sum'))
                     .round(2))

quality_summary


قد تكون كل الاعمال بدقة sd
لاكن ليس الكل بدقة hd
مثل الاعمال القديمه
المسلسلات الكلاسيكية
بعض المحتوى منخفض الميزانية
وهذا ممكن يكون سبب رئيسي في الفرق اللي شفناه.

In [ ]:
df['quality'].value_counts()
#عدد المشاهدات

رغم أن عدد برامج HD  إلا أن مشاهداته مرتفعة نسبيًا

In [ ]:
df['quality'] = df['hd'].map({1:'HD', 0:'SD'})
df[['program_name','quality']].drop_duplicates()['quality'].value_counts()
#عدد الأعمال المتوفرة لكل جودة

86% تقريبًا من المحتوى متوفر فقط بجودة SD

14% فقط متوفر بجودة HD

نلحظ ان اعمال منخفضة الدقة اكثر بي كثير من عاليه الدقه و هذا يفسر المشاهده العاليه للمنخفضة الدقه

In [ ]:
df.groupby('quality')['program_name'].nunique()


In [ ]:
df['year'] = df['date_'].dt.year
df.groupby(['year','quality'])['program_name'].nunique()
#التوزيع الزمني2017 2018

نلحظ قله الاعمال في 2018 عاليه الجوده بي شكل ملحوظ

In [ ]:
program_quality = df.groupby(['program_name','quality']).agg(
    total_hours=('duration_seconds', lambda x: x.sum()/3600)
).reset_index()


In [ ]:
program_quality.groupby('quality')['total_hours'].mean()
#متوسط ساعات المشاهدة لكل عمل

In [ ]:
# Here we try to get the most watched movies (Total Views / Total Users Views / Total watch time)
# For series we concatenated the Session episode to differentiate between episodes
grouped=df.copy()
grouped.loc[grouped['program_class'] == 'SERIES/EPISODES', 'program_name'] = grouped['program_name']+'_SE'+grouped['season'].astype(str)+'_EP'+grouped['episode'].astype(str)
grouped = grouped.groupby(['program_name','program_class'])\
.agg({'user_id_maped': [('co1', 'nunique'),('co2', 'count')],\
      'duration_seconds': [('co3', 'sum')] }).reset_index()
grouped.columns = ['program_name','program_class','No of Users who Watched', 'No of watches', 'Total watch time in seconds']
grouped['Total watch time in houres']=grouped['Total watch time in seconds']/3600
grouped = grouped.drop(columns=['Total watch time in seconds'])
grouped = grouped.sort_values(by=['Total watch time in houres', 'No of watches','No of Users who Watched'], ascending=False).reset_index(drop=True)


**رغم أن عالي الدقة يمثل 14% فقط من المحتوى، إلا أن معدل المشاهدة لكل عمل عالي الدقة يساوي أكثر من ضعف منخفض الدقة، مما يدل على أن جودة العرض تؤثر بشكل مباشر على عمق استهلاك المحتوى.**

In [ ]:
df=dataframe.copy()
df['quality'] = df['hd'].map({0:'SD', 1:'HD'})
df['hours']   = df['duration_seconds'] / 3600

In [ ]:
user_behavior = (
    df.groupby('user_id_maped')
      .agg(
          total_views=('program_name','size'),
          total_hours=('hours','sum'),
          unique_programs=('program_name','nunique'),
          hd_views=('hd','sum'),
          hd_rate=('hd','mean'),
          qualities_used=('quality','nunique')
      )
      .round(4)
      .reset_index()
)

user_behavior.head()


In [ ]:
user_behavior['quality_segment'] = np.where(
    user_behavior['qualities_used'] == 1,
    np.where(user_behavior['hd_rate'] == 1, 'HD_only', 'SD_only'),
    'Both'
)

user_behavior['quality_segment'].value_counts()
# تقسيم المستخدمين على حسب الجوده

In [ ]:
q1 = user_behavior['total_hours'].quantile(0.25)
q3 = user_behavior['total_hours'].quantile(0.75)

user_behavior['engagement_level'] = np.where(
    user_behavior['total_hours'] < q1, 'Light',
    np.where(user_behavior['total_hours'] > q3, 'Heavy', 'Medium')
)

user_behavior['engagement_level'].value_counts()


In [ ]:
user_behavior.sort_values('total_hours', ascending=False).head(10)


In [ ]:
segment_summary = (
    user_behavior.groupby('quality_segment')
      .agg(
          users=('user_id_maped','count'),
          avg_views=('total_views','mean'),
          avg_hours=('total_hours','mean'),
          avg_unique_programs=('unique_programs','mean'),
          median_hours=('total_hours','median')
      )
      .round(2)
)

segment_summary


In [ ]:
# show the result
grouped.head(35)

In [ ]:
# we import Visualization libraries
# you can ignore and use any other graphing libraries
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import plotly #a graphing library makes interactive, publication-quality graphs. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# plot top 10 Programs
fig = px.pie(grouped.head(10), values='Total watch time in houres', names='program_name',\
             hover_data=['program_class'],title='top 10 programs in total watch time in houres')
fig.show()

البرنامج الأعلى استحوذ على قرابة 19% من إجمالي المشاهدة داخل أفضل 10 أعمال، مما يدل على وجود محتوى "قائد".

الثلاثة الأوائل معًا يمثلون تقريبًا أكثر من 40% من إجمالي ساعات المشاهدة.

لا يوجد برنامج يهيمن بشكل ساحق، بل هناك توزيع نسبي متوازن بين الأعمال العشرة.

الفارق بين المركز الأول والمراكز الوسطى ليس كبيرًا جدًا، مما يشير إلى تنوع اهتمامات الجمهور.

وجود أعمال أنيميشن وعائلية ضمن الأعلى مشاهدة يدل على قوة هذا النوع من المحتوى.

الأفلام الطويلة غالبًا تحصل على ساعات مشاهدة أعلى بسبب مدة العرض مقارنة بالحلقات القصيرة.

In [ ]:
# Here we try to study the customer experience against Program class
grouped=df.copy()
grouped = grouped.groupby('program_class')\
.agg({'user_id_maped': [('co1', 'nunique'),('co2', 'count')],\
      'duration_seconds': [('co3', 'sum')] }).reset_index()
grouped.columns = ['program_class','No of Users who Watched', 'No of watches', 'Total watch time in seconds']
grouped['Total watch time in houres']=grouped['Total watch time in seconds']/3600
grouped = grouped.drop(columns=['Total watch time in seconds'])
grouped = grouped.sort_values(by=['Total watch time in houres', 'No of watches','No of Users who Watched'], ascending=False).reset_index(drop=True)


In [ ]:
df = dataframe.copy()

# حذف Column1 إذا موجود
if "Column1" in df.columns:
    df = df.drop(columns=["Column1"])

# تحويل hd إلى رقم (0/1)
df["hd"] = pd.to_numeric(df["hd"], errors="coerce")

# إنشاء عمود الجودة
df["quality"] = df["hd"].map({1:"HD", 0:"SD"})

# تحويل duration_seconds إلى رقم + إنشاء دقائق/ساعات
df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
df["minutes"] = df["duration_seconds"] / 60
df["hours"]   = df["duration_seconds"] / 3600

# ترتيب الاسماء (اختياري)
df["program_name"] = df["program_name"].astype(str).str.strip()

df.head()

In [ ]:
summary = df.groupby("quality").agg(
    views=("quality","size"),
    users=("user_id_maped","nunique"),
    mean_min=("minutes","mean"),
    total_hours=("hours","sum")
).round(2)

summary

In [ ]:
# Total hours by quality
df.groupby("quality")["hours"].sum().plot(kind="bar")
plt.title("Total Watch Hours (HD vs SD)")
plt.ylabel("Hours")
plt.xticks(rotation=0)
plt.show()

# Views by quality
df["quality"].value_counts().plot(kind="bar")
plt.title("Views Count (HD vs SD)")
plt.ylabel("Views")
plt.xticks(rotation=0)
plt.show()

In [ ]:
user_behavior = df.groupby("user_id_maped").agg(
    views=("user_id_maped","size"),
    hours=("hours","sum"),
    unique_programs=("program_name","nunique"),
    hd_rate=("hd","mean")      # نسبة استخدام HD
).round(3).reset_index()

user_behavior.head()

In [ ]:
user_behavior["segment"] = np.where(
    user_behavior["hd_rate"] == 1, "HD_only",
    np.where(user_behavior["hd_rate"] == 0, "SD_only", "Both")
)

user_behavior["segment"].value_counts()

In [ ]:
segment_stats = user_behavior.groupby("segment").agg(
    users=("user_id_maped","count"),
    avg_views=("views","mean"),
    avg_hours=("hours","mean"),
    avg_unique_programs=("unique_programs","mean")
).round(2)

segment_stats

In [ ]:
user_behavior.boxplot(column="unique_programs", by="segment")
plt.title("Unique Programs per User")
plt.suptitle("")
plt.ylabel("Unique programs")
plt.show()

In [ ]:
top_genres = (
    df.groupby(["quality","program_genre"])["hours"]
      .sum()
      .reset_index()
      .sort_values(["quality","hours"], ascending=[True, False])
)

top_hd = top_genres[top_genres["quality"]=="HD"].head(10)
top_sd = top_genres[top_genres["quality"]=="SD"].head(10)

top_hd, top_sd

In [ ]:
plt.figure(figsize=(10,4))
plt.bar(top_hd["program_genre"].astype(str), top_hd["hours"])
plt.title("Top 10 Genres in HD (Hours)")
plt.xticks(rotation=45, ha="right")
plt.show()

plt.figure(figsize=(10,4))
plt.bar(top_sd["program_genre"].astype(str), top_sd["hours"])
plt.title("Top 10 Genres in SD (Hours)")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
# show the result
grouped.head()

In [ ]:
# plot the total watch time against total number of users and report your findings
fig = px.pie(grouped, values='Total watch time in houres', names='program_class',\
             hover_data=['program_class'],title='Total duration spent by program_class')
fig2 = px.pie(grouped, values='No of Users who Watched', names='program_class',\
             hover_data=['program_class'],title='Total Users watching by program_class')

fig.update_traces(sort=False)
fig2.update_traces(sort=False)
fig.show()
fig2.show()

In [ ]:
# create HD usage flag per user
hd_user = df.groupby('user_id_maped')['hd'].mean().reset_index()

hd_user['hd_flag'] = hd_user['hd'].apply(
    lambda x: 'Uses_HD' if x > 0 else 'No_HD'
)

hd_user.head()

In [ ]:
user_behavior = user_behavior.merge(
    hd_user[['user_id_maped','hd_flag']],
    on='user_id_maped'
)

In [ ]:
comparison = user_behavior.groupby('hd_flag').agg(
    users=('user_id_maped','count'),
    avg_views=('views','mean'),
    avg_hours=('hours','mean'),
    avg_unique_programs=('unique_programs','mean')
).round(2)

comparison

1. قمنا بتقسيم المستخدمين حسب استخدامهم للجودة العالية: مستخدمون يشاهدون عالي الدقة، ومستخدمون لا يستخدمونها، إضافةً إلى فئة تستخدم الجودتين معًا.
2. من ناحية **المشاهدات**: المستخدمون الذين يجمعون بين الجودتين كانوا الأكثر تنفيذًا لعمليات مشاهدة، ما يعني أنهم يدخلون المنصة كثيرًا ويتنقلون بين محتويات متعددة.
3. من ناحية **الساعات**: إجمالي وقت المشاهدة كان أعلى بشكل واضح لدى مستخدمي الجودتين، ما يشير إلى أنهم يمضون وقتًا أطول داخل المنصة مقارنة ببقية الفئات.
4. من ناحية **التنوع**: فئة مستخدمي الجودتين شاهدت عددًا أكبر من البرامج المختلفة، وهذا يدل على فضول أعلى وتجربة محتوى أوسع.
5. من ناحية **التفاعل**: التفاعل الأعلى لا يرتبط بكون المستخدم “عالي الدقة فقط”، بل يرتبط بالمستخدم الذي يبدّل بين الجودتين حسب الموقف (سرعة الإنترنت/نوع المحتوى).
6. رغم أن المحتوى عالي الدقة أقل توفرًا من منخفض الدقة، إلا أن الأعمال المتاحة بجودة عالية تحقق **عمق مشاهدة أعلى لكل عمل** (وقت أطول لكل برنامج).
7. بشكل عملي: الجودة العالية تبدو عاملًا يزيد “عمق الاستهلاك”، لكن “المرونة في اختيار الجودة” هي التي تخلق أعلى ولاء وتفاعل.
8. التوصية: التركيز على دعم تجربة الاستخدام التي تسمح بالتبديل السلس بين الجودتين، وزيادة المحتوى عالي الدقة للأعمال الأكثر طلبًا، لأنها ترتبط بوقت مشاهدة أعمق وتحسين قيمة الاشتراك.
